# KV int4 Triton: sanity + speed vs bf16 `store_cache`

This notebook sanity-checks `quantized_set_kv_int4_triton` via **dequant round-trip** (packed cache + written scales vs original bf16) and benchmarks against JIT `store_cache` (`_set_kv_buffer_impl` bf16 path).

**Requirements:** CUDA GPU, `triton`, and an environment where `sglang` imports work (e.g. `pip install -e ./python` from this repo, or add `python/` to `PYTHONPATH`).

**Byte accounting (per timed call, conceptual):**
- **store_cache:** read `2×T×H×D×2` bf16 bytes + write the same volume into K/V cache slots.
- **int4 Triton:** read the same bf16 + write `2×T×H×(D/2)` packed uint8 + `2×T×H×2×4` fp32 scale/zero for K and V.

**Note:** `set_kv_buffer` may apply Hadamard before int4 in some configs; this notebook benchmarks the Triton quant kernel only (bf16 in → packed + scales out).

In [1]:
from __future__ import annotations

import os
import sys

import torch

# When running the notebook, set REPO_ROOT to your sglang-Rotation-fast checkout.
REPO_ROOT = os.environ.get("SGLANG_REPO_ROOT", os.getcwd())
PY_ROOT = os.path.join(REPO_ROOT, "python")
if os.path.isdir(PY_ROOT) and PY_ROOT not in sys.path:
    sys.path.insert(0, PY_ROOT)

assert torch.cuda.is_available(), "CUDA required"
DEVICE = torch.device("cuda")
torch.manual_seed(0)

from sglang.jit_kernel.kvcache import can_use_store_cache, store_cache
from sglang.srt.mem_cache.kv_quant_kernels import quantized_set_kv_int4_triton

## 1. Sanity: round-trip through packed int4

The kernel uses `(x).to(tl.uint8)` (truncate toward zero). PyTorch `floor(x)` can disagree with Triton when `v/scale+zero+0.5` is infinitesimally below an integer due to **different float32 FMA / eval order**, even though the **stored** `scale`/`zero` match a naive min–max recompute. So this notebook does **not** require bit-identical packed bytes vs a CPU reference.

The check below dequantizes using the **same** scale/zero the kernel wrote and compares to the original bf16 row (tolerances allow int4 error).

In [2]:
# Optional naive quant (debug only): may disagree with Triton on ~integer boundaries — see markdown above.
def ref_quantize_head_int4(x: torch.Tensor) -> tuple[torch.Tensor, float, float]:
    """x: float32 vector [head_dim]. Mirrors kernel math; packed bytes may still differ from Triton."""
    d = x.shape[0]
    assert d % 2 == 0
    half = d // 2
    v1, v2 = x[:half], x[half:]
    vmin = torch.minimum(v1.min(), v2.min())
    vmax = torch.maximum(v1.max(), v2.max())
    val_range = torch.maximum(vmax - vmin, torch.tensor(1e-8, device=x.device, dtype=x.dtype))
    scale = val_range / 15.0
    zero = -vmin / scale
    # Match tl: truncate float→uint8 (same as .to(uint8) for in-range positive values).
    t1 = (v1 / scale + zero + 0.5).to(torch.uint8)
    t2 = (v2 / scale + zero + 0.5).to(torch.uint8)
    packed = (t1 | (t2 << 4)).to(torch.uint8)
    return packed, float(scale.item()), float(zero.item())


def dequant_int4_packed(packed: torch.Tensor, scale: float, zero: float) -> torch.Tensor:
    """packed [..., D//2] -> float32 [..., D]"""
    q1 = (packed & 0x0F).to(torch.float32)
    q2 = ((packed >> 4) & 0x0F).to(torch.float32)
    d1 = (q1 - zero) * scale
    d2 = (q2 - zero) * scale
    return torch.cat([d1, d2], dim=-1)

In [3]:
def run_sanity(
    num_tokens: int = 32,
    num_heads: int = 8,
    head_dim: int = 128,
    cache_size: int = 4096,
) -> None:
    dtype = torch.bfloat16
    cache_k = torch.randn(num_tokens, num_heads, head_dim, device=DEVICE, dtype=dtype)
    cache_v = torch.randn(num_tokens, num_heads, head_dim, device=DEVICE, dtype=dtype)
    loc = torch.randperm(cache_size, device=DEVICE, dtype=torch.int32)[:num_tokens]

    k_cache = torch.zeros(cache_size, num_heads, head_dim // 2, device=DEVICE, dtype=torch.uint8)
    v_cache = torch.zeros_like(k_cache)
    k_sz = torch.zeros(cache_size, num_heads, 2, device=DEVICE, dtype=torch.float32)
    v_sz = torch.zeros_like(k_sz)

    quantized_set_kv_int4_triton(
        cache_k.clone(),
        cache_v.clone(),
        loc,
        k_cache,
        v_cache,
        k_sz,
        v_sz,
    )
    torch.cuda.synchronize()

    for ti in range(num_tokens):
        li = int(loc[ti].item())
        for h in range(num_heads):
            sk, zk = k_sz[li, h, 0].item(), k_sz[li, h, 1].item()
            sv, zv = v_sz[li, h, 0].item(), v_sz[li, h, 1].item()
            k_dq = dequant_int4_packed(k_cache[li, h : h + 1], sk, zk).squeeze(0).cpu()
            v_dq = dequant_int4_packed(v_cache[li, h : h + 1], sv, zv).squeeze(0).cpu()
            k_orig = cache_k[ti, h].float().cpu()
            v_orig = cache_v[ti, h].float().cpu()
            k_diff = (k_dq - k_orig).abs()
            v_diff = (v_dq - v_orig).abs()
            k_rel = k_diff / (k_orig.abs() + 1e-8)
            v_rel = v_diff / (v_orig.abs() + 1e-8)
            k_ok = (k_diff.max().item() < 0.3) or (k_rel.max().item() < 0.3)
            v_ok = (v_diff.max().item() < 0.3) or (v_rel.max().item() < 0.3)
            assert k_ok, f"K mismatch ti={ti} h={h} max_abs={k_diff.max():.4f} max_rel={k_rel.max():.4f}"
            assert v_ok, f"V mismatch ti={ti} h={h} max_abs={v_diff.max():.4f} max_rel={v_rel.max():.4f}"

    print(
        "Sanity OK: dequant(packed, scales from kernel) matches bf16 input within tolerance."
    )


run_sanity()

Sanity OK: dequant(packed, scales from kernel) matches bf16 input within tolerance.


### 1b. Fused Hadamard + int4 vs unfused rotation + int4

**Reference pipeline (strict):** float32 `× (1/√order)`, FWHT on each length-`order` block, **bf16 round-trip**, then `quantized_set_kv_int4_triton` — same as `test_fused_hadamard_int4_kv.py`. The fused kernel is expected to match this **bitwise** (`torch.equal` on packed bytes and scales). Both paths use the same Triton int4 launch settings (`num_warps=1`) so block min/max reductions match; restart the notebook kernel after `git pull` so you do not mix an old `kv_quant_kernels` with a new fused kernel.

**Unfused production path (optional):** `hadamard_transform(cache / √order)` from the `fast_hadamard_transform` package (same reshape as `memory_pool`), then the same int4 Triton write. If that package is not installed, the cell still checks fused vs the numpy reference only. Fused vs CUDA can differ bitwise in rare cases (see `fused_hadamard_int4_kv` docstring); the cell reports match or a short note.

In [4]:
import math

import numpy as np

try:
    from fast_hadamard_transform import hadamard_transform as _cuda_hadamard_transform
except ImportError:
    _cuda_hadamard_transform = None

from sglang.QuantKernel import quantized_set_kv_int4_hadamard_fused_triton
from sglang.srt.mem_cache.kv_quant_kernels import quantized_set_kv_int4_triton


def _numpy_fwht_f32(a: np.ndarray) -> np.ndarray:
    a2 = np.array(a, dtype=np.float32, copy=True)
    n = len(a2)
    h = 1
    while h < n:
        for i in range(0, n, h * 2):
            for j in range(h):
                u = a2[i + j]
                v = a2[i + j + h]
                a2[i + j] = u + v
                a2[i + j + h] = u - v
        h *= 2
    return a2


def ref_hadamard_bf16_pre_int4(x: torch.Tensor, order: int) -> torch.Tensor:
    """Same scaling + FWHT as fused / unit test: x.float() * (1/sqrt(order)) then FWHT, then bf16."""
    o = x.shape[-1]
    shaped = x.view(*x.shape[:-1], o // order, order)
    flat = shaped.float().reshape(-1, order) * (1.0 / math.sqrt(order))
    out = np.stack([_numpy_fwht_f32(flat[i].cpu().numpy()) for i in range(flat.shape[0])])
    y = torch.from_numpy(out).to(x.device).to(torch.bfloat16).reshape_as(shaped)
    return y.reshape_as(x)


def _apply_cuda_hadamard_bf16(x: torch.Tensor, order: int) -> torch.Tensor:
    """memory_pool unfused: view last dim as (…, order), hadamard_transform(x / sqrt(order)), restore."""
    if _cuda_hadamard_transform is None:
        raise RuntimeError("fast_hadamard_transform is not installed")
    orig = x.shape
    v = x.view(*orig[:-1], orig[-1] // order, order)
    v = _cuda_hadamard_transform(v / math.sqrt(order))
    return v.view(orig)


def run_sanity_rotation(
    num_tokens: int = 32,
    num_heads: int = 8,
    head_dim: int = 128,
    cache_size: int = 4096,
    hadamard_order: int = 16,
    rotate_v: bool = True,
) -> None:
    if head_dim % hadamard_order:
        raise ValueError(f"head_dim {head_dim} must be divisible by hadamard_order {hadamard_order}")
    dtype = torch.bfloat16
    cache_k = torch.randn(num_tokens, num_heads, head_dim, device=DEVICE, dtype=dtype)
    cache_v = torch.randn(num_tokens, num_heads, head_dim, device=DEVICE, dtype=dtype)
    loc = torch.randperm(cache_size, device=DEVICE, dtype=torch.int32)[:num_tokens]

    k_f = torch.zeros(cache_size, num_heads, head_dim // 2, device=DEVICE, dtype=torch.uint8)
    v_f = torch.zeros_like(k_f)
    ks_f = torch.zeros(cache_size, num_heads, 2, device=DEVICE, dtype=torch.float32)
    vs_f = torch.zeros_like(ks_f)

    quantized_set_kv_int4_hadamard_fused_triton(
        cache_k,
        cache_v,
        loc,
        k_f,
        v_f,
        ks_f,
        vs_f,
        hadamard_order,
        rotate_v=rotate_v,
    )
    torch.cuda.synchronize()

    k_ref = ref_hadamard_bf16_pre_int4(cache_k, hadamard_order)
    v_ref = (
        ref_hadamard_bf16_pre_int4(cache_v, hadamard_order)
        if rotate_v
        else cache_v
    )
    k_np = torch.zeros_like(k_f)
    v_np = torch.zeros_like(v_f)
    ks_np = torch.zeros_like(ks_f)
    vs_np = torch.zeros_like(vs_f)
    quantized_set_kv_int4_triton(k_ref, v_ref, loc, k_np, v_np, ks_np, vs_np)
    torch.cuda.synchronize()

    assert torch.equal(k_f, k_np), "K packed mismatch vs numpy FWHT + int4 reference"
    assert torch.equal(v_f, v_np), "V packed mismatch vs numpy FWHT + int4 reference"
    assert torch.equal(ks_f, ks_np), "K scales mismatch vs numpy FWHT + int4 reference"
    assert torch.equal(vs_f, vs_np), "V scales mismatch vs numpy FWHT + int4 reference"

    tag = f"order={hadamard_order} rotate_v={rotate_v}"
    print(f"Rotation sanity OK ({tag}): fused matches numpy-ref + int4 (bitwise).")

    if _cuda_hadamard_transform is None:
        print(
            "  Skipped CUDA unfused compare: `fast_hadamard_transform` not installed "
            "(install the package used by `memory_pool` to compare production unfused path)."
        )
        return

    k_cu = torch.zeros_like(k_f)
    v_cu = torch.zeros_like(v_f)
    ks_cu = torch.zeros_like(ks_f)
    vs_cu = torch.zeros_like(vs_f)
    k_in = _apply_cuda_hadamard_bf16(cache_k.clone(), hadamard_order)
    v_in = (
        _apply_cuda_hadamard_bf16(cache_v.clone(), hadamard_order)
        if rotate_v
        else cache_v.clone()
    )
    quantized_set_kv_int4_triton(k_in, v_in, loc, k_cu, v_cu, ks_cu, vs_cu)
    torch.cuda.synchronize()

    cuda_match = (
        torch.equal(k_f, k_cu)
        and torch.equal(v_f, v_cu)
        and torch.equal(ks_f, ks_cu)
        and torch.equal(vs_f, vs_cu)
    )
    if cuda_match:
        print("  Also bitwise-matches CUDA hadamard_transform + int4 (unfused production path).")
    else:
        print(
            "  CUDA unfused differs bitwise (expected in rare cases; see fused_hadamard_int4_kv docstring)."
        )


run_sanity_rotation(hadamard_order=16, rotate_v=True)
run_sanity_rotation(hadamard_order=16, rotate_v=False)

Rotation sanity OK (order=16 rotate_v=True): fused matches numpy-ref + int4 (bitwise).
  Also bitwise-matches CUDA hadamard_transform + int4 (unfused production path).
Rotation sanity OK (order=16 rotate_v=False): fused matches numpy-ref + int4 (bitwise).
  Also bitwise-matches CUDA hadamard_transform + int4 (unfused production path).


## 2. Speed: `quantized_set_kv_int4_triton` vs `store_cache`

Uses `triton.testing.do_bench` (ms). Adjust `NUM_TOKENS`, `NUM_HEADS`, `HEAD_DIM` to match your model. `store_cache` requires `row_bytes = NUM_HEADS * HEAD_DIM * 2` to be supported by the JIT (multiple of 4).

In [5]:
import triton.testing

NUM_TOKENS = 2048 * 8
NUM_HEADS = 32
HEAD_DIM = 256
CACHE_SIZE = 65536
DTYPE = torch.bfloat16

row_dim = NUM_HEADS * HEAD_DIM
row_bytes = row_dim * 2
print(f"row_bytes={row_bytes}, can_use_store_cache={can_use_store_cache(row_bytes)}")

loc = torch.randperm(CACHE_SIZE, device=DEVICE, dtype=torch.int32)[:NUM_TOKENS]

cache_k = torch.randn(NUM_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)
cache_v = torch.randn(NUM_TOKENS, NUM_HEADS, HEAD_DIM, device=DEVICE, dtype=DTYPE)

k_flat = cache_k.reshape(NUM_TOKENS, row_dim).contiguous()
v_flat = cache_v.reshape(NUM_TOKENS, row_dim).contiguous()
k_bf16_cache = torch.zeros(CACHE_SIZE, row_dim, device=DEVICE, dtype=DTYPE)
v_bf16_cache = torch.zeros_like(k_bf16_cache)

k_i4 = torch.zeros(CACHE_SIZE, NUM_HEADS, HEAD_DIM // 2, device=DEVICE, dtype=torch.uint8)
v_i4 = torch.zeros_like(k_i4)
k_sz = torch.zeros(CACHE_SIZE, NUM_HEADS, 2, device=DEVICE, dtype=torch.float32)
v_sz = torch.zeros_like(k_sz)

# Per-call batch (what each timed kernel touches for K+V this step)
n_elem_kv = NUM_TOKENS * NUM_HEADS * HEAD_DIM
bytes_k_in = n_elem_kv * 2  # bf16
bytes_v_in = n_elem_kv * 2
bytes_in = bytes_k_in + bytes_v_in
# store_cache: scatter bf16 into two cache tensors (one row each for K and V per token)
bytes_out_store = bytes_in
# int4: packed uint8 K+V + fp32 scale/zero per head for K and V
bytes_out_i4_pack = 2 * NUM_TOKENS * NUM_HEADS * (HEAD_DIM // 2)
bytes_out_i4_scales = 2 * NUM_TOKENS * NUM_HEADS * 2 * 4
bytes_out_i4 = bytes_out_i4_pack + bytes_out_i4_scales

print("\n=== Shapes (allocated buffers) ===")
print(f"  cache_k / cache_v: {tuple(cache_k.shape)}  dtype={DTYPE}")
print(f"  k_bf16_cache:      {tuple(k_bf16_cache.shape)}  (only {NUM_TOKENS} rows written per call)")
print(f"  k_i4 / v_i4:       {tuple(k_i4.shape)}  uint8 packed")
print(f"  k_sz / v_sz:       {tuple(k_sz.shape)}  float32 [scale, zero]")
print("\n=== Per-call traffic (this batch, for throughput denominator) ===")
print(f"  Input read:        {bytes_in / 1e6:.4f} MB  (K bf16 {bytes_k_in/1e6:.4f} + V bf16 {bytes_v_in/1e6:.4f})")
print(f"  store_cache write: {bytes_out_store / 1e6:.4f} MB  (bf16 K row + V row @ loc)")
print(f"  int4 write:        {bytes_out_i4 / 1e6:.4f} MB  (packed {bytes_out_i4_pack/1e6:.4f} + scales {bytes_out_i4_scales/1e6:.4f})")
print(f"  int4 vs store write ratio: {bytes_out_i4 / bytes_out_store:.3f}x  (less bytes out; more compute)\n")


def bench_store():
    store_cache(
        k_flat,
        v_flat,
        k_bf16_cache,
        v_bf16_cache,
        loc,
        row_bytes=row_bytes,
    )


def bench_int4():
    quantized_set_kv_int4_triton(
        cache_k,
        cache_v,
        loc,
        k_i4,
        v_i4,
        k_sz,
        v_sz,
    )


for fn in (bench_store, bench_int4):
    fn()
torch.cuda.synchronize()

ms_store = triton.testing.do_bench(bench_store, rep=20)
ms_int4 = triton.testing.do_bench(bench_int4, rep=20)

bytes_store = bytes_in + bytes_out_store
bytes_int4 = bytes_in + bytes_out_i4


def gbps(ms: float, nbytes: float) -> float:
    return (nbytes / 1e9) / (ms / 1e3)


print("=== Timing (read + write as above) ===")
print(
    f"store_cache:     {ms_store:.3f} ms/call  (~{gbps(ms_store, bytes_store):.2f} GB/s  @ {bytes_store/1e6:.4f} MB)"
)
print(
    f"int4 triton:     {ms_int4:.3f} ms/call  (~{gbps(ms_int4, bytes_int4):.2f} GB/s  @ {bytes_int4/1e6:.4f} MB)"
)
print("(GB/s = (input+output bytes) / time; actual DRAM/L2 traffic may differ.)")

row_bytes=16384, can_use_store_cache=True

=== Shapes (allocated buffers) ===
  cache_k / cache_v: (16384, 32, 256)  dtype=torch.bfloat16
  k_bf16_cache:      (65536, 8192)  (only 16384 rows written per call)
  k_i4 / v_i4:       (65536, 32, 128)  uint8 packed
  k_sz / v_sz:       (65536, 32, 2)  float32 [scale, zero]

=== Per-call traffic (this batch, for throughput denominator) ===
  Input read:        536.8709 MB  (K bf16 268.4355 + V bf16 268.4355)
  store_cache write: 536.8709 MB  (bf16 K row + V row @ loc)
  int4 write:        142.6063 MB  (packed 134.2177 + scales 8.3886)
  int4 vs store write ratio: 0.266x  (less bytes out; more compute)

=== Timing (read + write as above) ===
store_cache:     0.376 ms/call  (~2856.05 GB/s  @ 1073.7418 MB)
int4 triton:     0.644 ms/call  (~1055.42 GB/s  @ 679.4772 MB)
(GB/s = (input+output bytes) / time; actual DRAM/L2 traffic may differ.)


## 3. Throughput: plain int4 vs fused Hadamard + int4 (multiple orders)

**What is timed:** same bf16 `cache_k` / `cache_v` read and same packed + scale/zero writes as §2. The fused kernel also uses preallocated fp32 scratch `[T, H, D, 2]` for K and V (ping-pong FWHT); we **preallocate** it so `do_bench` does not measure Python alloc.

**GB/s here:** `(bytes read K+V bf16 + bytes written packed + scales) / time`, same denominator as the int4 row in §2 for comparability. Actual on-chip traffic is higher for the fused path due to scratch.

**Config:** `D_THRUPT` must be divisible by every entry in `HADAMARD_ORDERS` (default `16, 32, 64, 128`).

In [7]:
import triton.testing

from sglang.QuantKernel import quantized_set_kv_int4_hadamard_fused_triton
from sglang.srt.mem_cache.kv_quant_kernels import quantized_set_kv_int4_triton

# --- Edit shapes here ---
T_THRUPT = 2048 * 8
H_THRUPT = 32
D_THRUPT = 256
CACHE_THRUPT = 65536
HADAMARD_ORDERS = (16, 32, 64, 128, 256)
DTYPE_BT = torch.bfloat16
REP = 20

for ho in HADAMARD_ORDERS:
    assert D_THRUPT % ho == 0, f"D_THRUPT={D_THRUPT} must be divisible by hadamard_order={ho}"

loc_t = torch.randperm(CACHE_THRUPT, device=DEVICE, dtype=torch.int32)[:T_THRUPT]
ck = torch.randn(T_THRUPT, H_THRUPT, D_THRUPT, device=DEVICE, dtype=DTYPE_BT)
cv = torch.randn(T_THRUPT, H_THRUPT, D_THRUPT, device=DEVICE, dtype=DTYPE_BT)

k_i4_t = torch.zeros(CACHE_THRUPT, H_THRUPT, D_THRUPT // 2, device=DEVICE, dtype=torch.uint8)
v_i4_t = torch.zeros_like(k_i4_t)
k_sz_t = torch.zeros(CACHE_THRUPT, H_THRUPT, 2, device=DEVICE, dtype=torch.float32)
v_sz_t = torch.zeros_like(k_sz_t)

work_k = torch.empty(T_THRUPT, H_THRUPT, D_THRUPT, 2, device=DEVICE, dtype=torch.float32)
work_v = torch.empty_like(work_k)

n_elem = T_THRUPT * H_THRUPT * D_THRUPT
bytes_in_kv = 2 * n_elem * 2
bytes_out_i4 = (
    2 * T_THRUPT * H_THRUPT * (D_THRUPT // 2)
    + 2 * T_THRUPT * H_THRUPT * 2 * 4
)
bytes_total = bytes_in_kv + bytes_out_i4
scratch_bytes = 2 * T_THRUPT * H_THRUPT * D_THRUPT * 2 * 4  # work_k + work_v (not in GB/s denom below)


def _gbps(ms: float, nbytes: float) -> float:
    return (nbytes / 1e9) / (ms / 1e3)


def bench_plain_int4():
    quantized_set_kv_int4_triton(ck, cv, loc_t, k_i4_t, v_i4_t, k_sz_t, v_sz_t)


results = []

ms_plain = triton.testing.do_bench(bench_plain_int4, rep=REP)
results.append(
    ("int4 (no rotation)", "-", ms_plain, _gbps(ms_plain, bytes_total))
)

for ho in HADAMARD_ORDERS:
    for rot_v in (True, False):
        label = f"fused H×int4  order={ho}  rotate_v={rot_v}"

        def _run(ho_=ho, rv=rot_v):
            quantized_set_kv_int4_hadamard_fused_triton(
                ck,
                cv,
                loc_t,
                k_i4_t,
                v_i4_t,
                k_sz_t,
                v_sz_t,
                hadamard_order=ho_,
                work_k=work_k,
                work_v=work_v,
                rotate_v=rv,
            )

        ms = triton.testing.do_bench(lambda: _run(), rep=REP)
        results.append((label, ho, ms, _gbps(ms, bytes_total)))

torch.cuda.synchronize()

print(f"Shapes: T={T_THRUPT} H={H_THRUPT} D={D_THRUPT}  |  bytes/call (K+V in + packed+sz out)={bytes_total/1e6:.4f} MB")
print(f"Fused scratch (preallocated, not in GB/s column): {scratch_bytes/1e6:.4f} MB  (work_k + work_v fp32)\n")
print(f"{'case':<42} {'order':>6} {'ms/call':>10} {'GB/s':>10}")
print("-" * 72)
for name, order, ms, g in results:
    o = "-" if order == "-" else str(order)
    print(f"{name:<42} {o:>6} {ms:>10.4f} {g:>10.2f}")

Shapes: T=16384 H=32 D=256  |  bytes/call (K+V in + packed+sz out)=679.4772 MB
Fused scratch (preallocated, not in GB/s column): 2147.4836 MB  (work_k + work_v fp32)

case                                        order    ms/call       GB/s
------------------------------------------------------------------------
int4 (no rotation)                              -     0.6391    1063.20
fused H×int4  order=16  rotate_v=True          16     1.1673     582.10
fused H×int4  order=16  rotate_v=False         16     0.9033     752.18
fused H×int4  order=32  rotate_v=True          32     1.2738     533.45
fused H×int4  order=32  rotate_v=False         32     0.9565     710.36
fused H×int4  order=64  rotate_v=True          64     1.2871     527.93
fused H×int4  order=64  rotate_v=False         64     0.9639     704.95
fused H×int4  order=128  rotate_v=True        128     1.3241     513.16
fused H×int4  order=128  rotate_v=False       128     0.9819     692.04
fused H×int4  order=256  rotate_v=True  